
# ETL de Ejemplo con Jupyter Notebook — Sakila (SQLite)

**Objetivo:** Demostrar un flujo **ETL** completo dentro de un Notebook:
1) **Extract**: Conectar a SQLite (Sakila) y extraer datos con SQL.  
2) **Transform**: Agregar/filtrar resultados con Pandas (post-procesado ligero).  
3) **Load**: Visualizar (Seaborn) y **exportar** resultados finales a Excel (y figura).  


## 0) Parámetros y configuración

In [ ]:

from pathlib import Path
from datetime import datetime

# Parámetros de salida
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TS = datetime.now().strftime("%Y%m%d-%H%M%S")

# Ruta a la base SQLite
DB_URL = "sqlite:///../../data/sqlite-sakila.db"
DB_URL


## 1) Imports

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import sqlalchemy as sa
import seaborn as sns
import pandas as pd


pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

sns.set_theme(context="notebook", style="whitegrid", font_scale=1.0)
print("Versions → pandas:", pd.__version__)
print("Versions → seaborn:", sns.__version__)
print("Versions → sqlalchemy:", sa.__version__)

## 2) Extract — Conexión a SQLite (Sakila) y SQL

In [ ]:
engine = sa.create_engine(DB_URL)

SQL_TOP = """
SELECT 
    f.film_id,
    f.title,
    COUNT(r.rental_id) AS rentals
FROM film AS f
JOIN inventory AS i  ON i.film_id = f.film_id
JOIN rental   AS r  ON r.inventory_id = i.inventory_id
GROUP BY f.film_id, f.title
"""

df_ = pd.read_sql_query(SQL_TOP, con=engine)
df_


## 3) Transform — Post-procesado ligero

In [ ]:
df_["rentals"] = pd.to_numeric(df_["rentals"], errors="coerce").astype("Int64")
df_top10 = (
    df_
    .sort_values(["rentals", "title"], ascending=[False, True])
    .head(10)
    .assign(rank=lambda x: range(1, len(x) + 1))
    .reset_index(drop=True)
)
df_top10

## 4) Visualización — Seaborn (Horizontal Bar Chart)

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=df_top10, y="title", x="rentals")
ax.set_title("Top 10 Películas por Número de Rentas — Sakila (SQLite)")
ax.set_xlabel("Número de rentas")
ax.set_ylabel("Película")
plt.tight_layout()

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from datetime import datetime
TS = datetime.now().strftime("%Y%m%d-%H%M%S")
fig_path = OUTPUT_DIR / f"sakila_top10_rentals_{TS}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figura exportada a:", fig_path)


## 5) Load — Exportar resultados

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TS = datetime.now().strftime("%Y%m%d-%H%M%S")

excel_path = OUTPUT_DIR / f"sakila_top10_rentals_{TS}.xlsx"
csv_path   = OUTPUT_DIR / f"sakila_top10_rentals_{TS}.csv"

df_top10.to_excel(excel_path, sheet_name="Top10_Rentals", index=False)


## 6) Exportar el Notebook a PDF (opcional)

Para generar un PDF a partir de este notebook:

```bash
jupyter nbconvert --to pdf ETL_Sakila.ipynb
```
Si no tienes LaTeX, exporta a **HTML** y luego imprime a PDF:
```bash
jupyter nbconvert --to html ETL_Sakila.ipynb
```



---

### ✅ Checklist del ETL
- **Extract:** SQL → Pandas con SQLAlchemy.  
- **Transform:** Orden y ranking en Pandas.  
- **Load:** Visualización (Seaborn) + Exportación Excel/CSV (+ imagen).  
